# Phase 1: Data Acquisition & Inspection

**Project:** Bureau of Tax and Economic Analysis — Quantitative Research

**Purpose:** Source, inspect, validate, and export all data needed for Q1 and Q2 analysis.

**Data Sources:**
1. NYS DOL CES data (`ces.csv` from https://dol.ny.gov/statistics-ceszip)
2. NYC OpenData 311 Service Requests (API at https://data.cityofnewyork.us/resource/erm2-nwe9.json)

In [1]:
import pandas as pd
import requests
import os

DATA_DIR = '../data'

---
## Step 1.1: Load and Inspect CES Data


In [2]:
ces = pd.read_csv(f'{DATA_DIR}/ces.csv')
ces.columns = ces.columns.str.strip()

print(f'Shape: {ces.shape[0]:,} rows × {ces.shape[1]} columns')
print(f'Columns: {list(ces.columns)}')
print()
print('=== HEAD ===')
display(ces.head(3))
print('=== TAIL ===')
display(ces.tail(3))

Shape: 23,810 rows × 19 columns
Columns: ['SERIESCODE', 'INDUSTRY_TITLE', 'AREATYPE', 'AREA', 'AREANAME', 'YEAR', 'JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC', 'ANNUAL']

=== HEAD ===


,SERIESCODE,INDUSTRY_TITLE,AREATYPE,AREA,AREANAME,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANNUAL
0,0,Total Nonfarm,1,36,New York State,1990,8093100,8121400,8180500.0,8187500.0,8283200.0,8333300.0,8202400.0,8206000.0,8213500.0,8211100.0,8211200.0,8201800.0,8203800 ...
1,0,Total Nonfarm,1,36,New York State,1991,7841400,7839000,7870500.0,7878900.0,7934800.0,7986600.0,7833400.0,7834400.0,7844300.0,7886700.0,7904100.0,7888700.0,7878600 ...
2,0,Total Nonfarm,1,36,New York State,1992,7597100,7611300,7648100.0,7701800.0,7762200.0,7803900.0,7724000.0,7713500.0,7726100.0,7781200.0,7790500.0,7807400.0,7722300 ...


=== TAIL ===


,SERIESCODE,INDUSTRY_TITLE,AREATYPE,AREA,AREANAME,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANNUAL
23807,90932622,Local Government Hospitals,23,35004,Nassau-Suffolk Metropolitan Division,2024,2800,2800,2800.0,2800.0,2800.0,2800.0,2800.0,2800.0,2800.0,2800.0,2800.0,2800.0,2800 ...
23808,90932622,Local Government Hospitals,23,35004,Nassau-Suffolk Metropolitan Division,2025,3100,2800,2800.0,2900.0,2900.0,2900.0,2900.0,2900.0,2900.0,2800.0,2800.0,2900.0,2900 ...
23809,90932622,Local Government Hospitals,23,35004,Nassau-Suffolk Metropolitan Division,2026,2900,2900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...


### Data Dictionary

Source: `data/CES_Readme.txt` — included in the ZIP download from https://dol.ny.gov/statistics-ceszip

| Field | Definition |
|-------|------------|
| `SERIESCODE` | CES published line code |
| `INDUSTRY_TITLE` | CES published line name |
| `AREATYPE` | Area type (01=state, 21=MSA, 23=MD) |
| `AREA` | Numeric metro area FIPS code |
| `AREANAME` | Name of area |
| `YEAR` | Year |
| `JAN`–`DEC` | Monthly employment |
| `ANNUAL` | Summed monthly employment / 12 |

**Key notes from the README:**
- Data is **not seasonally adjusted** — compare same month across years only
- Values are in **actual units** (not thousands)

---
## Step 1.2: Verify Geography — How is NYC Represented?


In [3]:
area_names = sorted(ces['AREANAME'].unique())
print(f'Total unique areas: {len(area_names)}')
print(f'NYC entry: {[a for a in area_names if "New York City" in a]}')

nyc = ces[ces['AREANAME'] == 'New York City']
tnf = nyc[nyc['INDUSTRY_TITLE'] == 'Total Nonfarm']
latest_tnf = tnf[tnf['YEAR'] == tnf['YEAR'].max()]
max_year = tnf['YEAR'].max()
feb_val = latest_tnf['FEB'].values[0]
print(f'NYC Total Nonfarm (Feb {max_year}): {feb_val:,}')


Total unique areas: 15
NYC entry: ['New York City']
NYC Total Nonfarm (Feb 2026): 4,791,000


---
## Step 1.3: Verify Latest Available Month


In [4]:
max_year = ces['YEAR'].max()
latest_year_df = ces[ces['YEAR'] == max_year]
print(f'Max year in data: {max_year}')
print(f'\nMonth columns with non-null data in {max_year}:')

months = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']
status_data = []
for m in months:
    non_null = latest_year_df[m].notna().sum()
    status_data.append({'Month': m, 'Non-null rows': non_null, 'Status': 'HAS DATA' if non_null > 0 else 'NO DATA'})
display(pd.DataFrame(status_data))

latest_months = [m for m in months if latest_year_df[m].notna().sum() > 0]
month_names = {'JAN':'January','FEB':'February','MAR':'March','APR':'April','MAY':'May','JUN':'June',
               'JUL':'July','AUG':'August','SEP':'September','OCT':'October','NOV':'November','DEC':'December'}
latest_month_name = month_names[latest_months[-1]]
prev_year = max_year - 1
baseline_year = max_year - 5
print(f'Latest available month: {latest_month_name} {max_year}')
print(f'  -> YoY: {latest_month_name} {max_year} vs {latest_month_name} {prev_year}')
print(f'  -> 5yr: {latest_month_name} {max_year} vs {latest_month_name} {baseline_year} (COVID-distorted baseline)')


Max year in data: 2026

Month columns with non-null data in 2026:


,Month,Non-null rows,Status
0,JAN,650,HAS DATA
1,FEB,650,HAS DATA
2,MAR,0,NO DATA
3,APR,0,NO DATA
4,MAY,0,NO DATA
5,JUN,0,NO DATA
6,JUL,0,NO DATA
7,AUG,0,NO DATA
8,SEP,0,NO DATA
9,OCT,0,NO DATA


Latest available month: February 2026
  -> YoY: February 2026 vs February 2025
  -> 5yr: February 2026 vs February 2021 (COVID-distorted baseline)


---
## Step 1.4: BLS Supersector Industry Mapping

Reference: https://www.bls.gov/sae/additional-resources/naics-supersectors-for-ces-program.htm

The BLS defines **10 supersectors** that partition all private-sector employment plus Government.
These 10 supersectors **sum exactly to Total Nonfarm** with no overlap — each is a direct CES series.

**Why we use 10 BLS supersectors instead of 18 individual NAICS 2-digit sectors:**

1. **BLS standard:** NYS DOL, BLS, and the NYC Comptroller all present NYC employment using these supersectors.
2. **No double-counting:** The 10 supersectors partition Total Nonfarm exactly. Five supersectors aggregate
   smaller 2-digit sectors (e.g., Leisure & Hospitality = NAICS 71 + 72), and those component sectors are
   available as separate CES series — but presenting both levels would be redundant.
3. **Statistical robustness:** Individual sub-sectors like Utilities (17K) and Management of Companies (80K)
   are too small for meaningful YoY trend analysis.
4. **Analytical clarity:** 10 sectors + Total Nonfarm keeps tables and charts readable; 18 sectors dilute focus.

**Note:** Education (NAICS 61) is private-sector only. Government (NAICS 92) is shown separately.


In [5]:
supersectors = [
    ('21+23', 'Mining, Logging and Construction'),
    ('31-33', 'Manufacturing'),
    ('22,42-49', 'Trade, Transportation, and Utilities'),
    ('51', 'Information'),
    ('52,53', 'Financial Activities'),
    ('54-56', 'Professional and Business Services'),
    ('61,62', 'Private Education and Health Services'),
    ('71,72', 'Leisure and Hospitality'),
    ('81', 'Other Services'),
    ('92', 'Government'),
]

# Sub-sector composition for the 5 aggregated supersectors
composition = {
    '22,42-49': ['22: Utilities', '42: Wholesale Trade', '44-45: Retail Trade', '48-49: Transport/Warehousing'],
    '52,53': ['52: Finance/Insurance', '53: Real Estate'],
    '54-56': ['54: Prof/Technical Svcs', '55: Management of Cos', '56: Admin/Waste Mgmt'],
    '61,62': ['61: Education (Private)', '62: Health Care/Social Asst'],
    '71,72': ['71: Arts/Entertainment', '72: Accommodation/Food'],
}

nyc_industries = set(ces[ces['AREANAME'] == 'New York City']['INDUSTRY_TITLE'].unique())

rows = []
for naics, title in supersectors:
    in_data = title in nyc_industries
    sub = ', '.join(composition.get(naics, []))
    rows.append({'NAICS': naics, 'Supersector': title, 'Sub-sectors': sub if sub else '(standalone)', 'In CES Data': in_data})

rows.append({'NAICS': '', 'Supersector': 'Total Nonfarm', 'Sub-sectors': 'Sum of all 10 above', 'In CES Data': 'Total Nonfarm' in nyc_industries})

df_map = pd.DataFrame(rows)
df_map['In CES Data'] = df_map['In CES Data'].map({True: 'Yes', False: 'No'})
display(df_map.style.hide(axis='index'))

print(f'All 10 supersectors + Total Nonfarm confirmed in CES data.')
print(f'5 of 10 supersectors are aggregations — their sub-sectors are also available individually in CES.')


NAICS,Supersector,Sub-sectors,In CES Data
21+23,"Mining, Logging and Construction",(standalone),Yes
31-33,Manufacturing,(standalone),Yes
"22,42-49","Trade, Transportation, and Utilities","22: Utilities, 42: Wholesale Trade, 44-45: Retail Trade, 48-49: Transport/Warehousing",Yes
51,Information,(standalone),Yes
"52,53",Financial Activities,"52: Finance/Insurance, 53: Real Estate",Yes
54-56,Professional and Business Services,"54: Prof/Technical Svcs, 55: Management of Cos, 56: Admin/Waste Mgmt",Yes
"61,62",Private Education and Health Services,"61: Education (Private), 62: Health Care/Social Asst",Yes
"71,72",Leisure and Hospitality,"71: Arts/Entertainment, 72: Accommodation/Food",Yes
81,Other Services,(standalone),Yes
92,Government,(standalone),Yes


All 10 supersectors + Total Nonfarm confirmed in CES data.
5 of 10 supersectors are aggregations — their sub-sectors are also available individually in CES.


---
## Step 1.5: Explore 311 API — Complaint Types

Before building the main query, we explore the API to verify exact field values.


In [6]:
API_BASE = 'https://data.cityofnewyork.us/resource/erm2-nwe9.json'

params = {
    '$select': 'complaint_type, count(*) as cnt',
    '$where': "created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' "
             "AND (complaint_type like '%Noise%' OR complaint_type like '%Illegal Parking%')",
    '$group': 'complaint_type',
    '$order': 'cnt DESC',
    '$limit': 100
}
r = requests.get(API_BASE, params=params)
noise_df = pd.DataFrame(r.json())
noise_df['cnt'] = noise_df['cnt'].astype(int)
display(noise_df.style.hide(axis='index'))

total_noise = noise_df[noise_df['complaint_type'].str.contains('Noise')]['cnt'].sum()
exact_noise = noise_df[noise_df['complaint_type'] == 'Noise']['cnt'].sum()
print(f'All noise subcategories: {total_noise:,}')
print(f'Exact "Noise" only:      {exact_noise:,}')
print(f'Ratio: {total_noise/exact_noise:.1f}x more captured with starts_with approach')

complaint_type,cnt
Illegal Parking,86493
Noise - Residential,75543
Noise - Street/Sidewalk,12743
Noise - Vehicle,11372
Noise - Commercial,11114
Noise,10789
Noise - Helicopter,5290
Noise - Park,498
Noise - House of Worship,395


All noise subcategories: 127,744
Exact "Noise" only:      10,789
Ratio: 11.8x more captured with starts_with approach


### Noise Matching Decision

**The project says: `complaint_type = noise or illegal parking`**

**Our interpretation:** Capture ALL complaint types that start with "Noise" (including subcategories) plus "Illegal Parking" exactly.

**Justification:**
1. The 311 system categorizes noise into subcategories (Residential, Commercial, Vehicle, etc.)
2. An exact match for "Noise" alone captures only ~8% of noise complaints
3. The exact "Noise" type belongs to **DEP** (not NYPD or HPD), so it is correctly excluded by the agency filter
4. HPD has zero noise/parking complaints — HPD handles housing quality (heat/hot water, plumbing)

---
## Step 1.6: Agency Breakdown — NYPD vs HPD

Verify which agencies handle noise/parking complaints. HPD is included in the query but expected to return zero rows.


In [7]:
params = {
    '$select': 'agency_name, count(*) as cnt',
    '$where': "created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' "
             "AND (starts_with(complaint_type, 'Noise') "
             "OR complaint_type='Illegal Parking') "
             "AND (agency_name='New York City Police Department' "
             "OR agency_name='Department of Housing Preservation and Development')",
    '$group': 'agency_name',
    '$order': 'cnt DESC',
    '$limit': 100
}
r = requests.get(API_BASE, params=params)
agency_df = pd.DataFrame(r.json())
if 'cnt' in agency_df.columns:
    agency_df['cnt'] = agency_df['cnt'].astype(int)

# Ensure HPD appears as a line item even if count is zero
hpd_row = agency_df[agency_df['agency_name'] == 'Department of Housing Preservation and Development']
if len(hpd_row) == 0:
    agency_df = pd.concat([agency_df, pd.DataFrame([{
        'agency_name': 'Department of Housing Preservation and Development',
        'cnt': 0
    }])], ignore_index=True)

agency_df = agency_df.sort_values('cnt', ascending=False).reset_index(drop=True)
display(agency_df.style.hide(axis='index'))

nypd_cnt = agency_df[agency_df['agency_name'].str.contains('Police')]['cnt'].sum()
hpd_cnt = agency_df[agency_df['agency_name'].str.contains('Housing')]['cnt'].sum()
print(f'NYPD: {nypd_cnt:,} complaints')
print(f'HPD:  {hpd_cnt:,} complaints (zero — HPD handles housing quality: heat/hot water, plumbing, not noise/parking)')


agency_name,cnt
New York City Police Department,198158
Department of Housing Preservation and Development,0


NYPD: 198,158 complaints
HPD:  0 complaints (zero — HPD handles housing quality: heat/hot water, plumbing, not noise/parking)


---
## Step 1.7: Build Full 311 Query and Pull Data


In [8]:
select_fields = 'unique_key,created_date,agency_name,complaint_type,descriptor,borough,incident_zip,latitude,longitude'
where_clause = (
    "created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' "
    "AND (agency_name='New York City Police Department' "
    "OR agency_name='Department of Housing Preservation and Development') "
    "AND (starts_with(complaint_type, 'Noise') "
    "OR complaint_type='Illegal Parking')"
)

params = {
    '$select': select_fields,
    '$where': where_clause,
    '$limit': 500000
}

r = requests.get(API_BASE, params=params)
print(f'Status: {r.status_code}')
df_311 = pd.DataFrame(r.json())
print(f'Rows pulled: {len(df_311):,}')

Status: 200


Rows pulled: 198,158


In [9]:
summary = {
    'Total rows': f'{len(df_311):,}',
    'Date range': f'{df_311["created_date"].min()} to {df_311["created_date"].max()}',
    'Agencies': ', '.join(sorted(df_311['agency_name'].unique())),
    'Complaint types': ', '.join(sorted(df_311['complaint_type'].unique())),
    'Duplicate unique_key': f'{df_311["unique_key"].duplicated().sum()}',
}
display(pd.DataFrame(list(summary.items()), columns=['Check', 'Value']).style.hide(axis='index'))


Check,Value
Total rows,"198,158"
Date range,2021-12-15T00:02:21.000 to 2022-03-15T23:59:38.000
Agencies,New York City Police Department
Complaint types,"Illegal Parking, Noise - Commercial, Noise - House of Worship, Noise - Park, Noise - Residential, Noise - Street/Sidewalk, Noise - Vehicle"
Duplicate unique_key,0


In [10]:
output_path = f'{DATA_DIR}/311_complaints.csv'
df_311.to_csv(output_path, index=False)
print(f'Exported {len(df_311):,} rows → {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1024:.1f} KB')

Exported 198,158 rows → ../data/311_complaints.csv
File size: 29659.0 KB


---
## Summary

### CES Employment Data (Q1)
- **Geography:** `AREANAME == 'New York City'` (direct entry, not the broader MSA)
- **Latest month:** February 2026
- **Industry classification:** 10 BLS supersectors + Total Nonfarm (see Step 1.4 rationale)
- **Data is NSA:** compare same month across years only
- **5-year baseline (Feb 2021) is COVID-distorted** — must be central to Q1b narrative

### 311 Service Requests (Q2)
- **Total rows:** 198,158 (all NYPD, 0 HPD)
- **HPD has zero matching rows:** HPD handles housing quality, not noise/parking — valid finding, shown explicitly in Step 1.6 table
- **Noise matching:** `starts_with(complaint_type, 'Noise')` captures all subcategories; exact "Noise" is DEP-only
- **No duplicates, dates inclusive on both ends**

Full audit details: `audit_phase1.md`
